# traineo1 Kaggle Benchmark Clean

Notebook ini meniru pola `traineo1_kaggle_clean`, tetapi difokuskan untuk:
- training **baseline F5-TTS v1 Base** di **2x Kaggle T4**,
- training **hybrid F5-TTS + Mamba** di **2x Kaggle T4**,
- **tanpa fallback** untuk jalur hybrid: `flash_attn` wajib aktif,
- benchmark compute baseline vs hybrid,
- evaluasi resmi yang relevan dengan paper: **WER** dan **SIM-o**,
- menyiapkan bundle **SMOS** dan **CMOS** untuk penilaian manusia.

Catatan penting:
- **SMOS** dan **CMOS** tidak bisa dihitung valid tanpa penilai manusia. Notebook ini menyiapkan manifest buta dan template rating agar kedua model tetap menghasilkan paket evaluasi yang sama.
- Notebook ini **hard fail** kalau tidak mendeteksi **2 GPU**. Tidak ada fallback ke single GPU.
- Baseline tetap pakai `attn_backend=torch`.
- Hybrid dipaksa pakai `attn_backend=flash_attn` agar beda praktisnya terlihat.


In [ ]:
# Cell 1: Repo + benchmark config
REPO_URL = "https://github.com/AneKazek/gardenbunga.git"
REPO_BRANCH = "main"
REPO_NAME = "gardenbunga"
NOTEBOOK_TAG = "traineo1_kaggle_benchmark"

DATASET_NAME = "tts-indo"
DATASET_ROOT_RAW = "/kaggle/input/datasets/benedictusryugunawan/tts-indo"
CSV_1_RAW = "/kaggle/input/datasets/benedictusryugunawan/tts-indo/data/metadata.csv"
CSV_2_RAW = "/kaggle/input/datasets/benedictusryugunawan/tts-indo/data/metadata_indsp.csv"

# Isi path lokal Kaggle Input untuk evaluasi.
EVAL_SPECS = [
    {
        "name": "seedtts_test_en",
        "task_type": "seedtts",
        "lang": "en",
        "meta_file": "/kaggle/input/seed-tts-eval/seedtts_testset/en/meta.lst",
        "librispeech_test_clean_path": None,
    },
]
WAVLM_CKPT_RAW = "/kaggle/input/wavlm-large-finetune/wavlm_large_finetune.pth"

PRETRAIN_CKPT = "hf://SWivid/F5-TTS/F5TTS_v1_Base/model_1250000.safetensors"
HF_TOKEN_RAW = ""

ACCELERATE_NUM_PROCESSES = 2
ACCELERATE_MIXED_PRECISION = "fp16"
TRAIN_NUM_WORKERS = 4
TRAIN_EPOCHS = 11
TRAIN_LR = 7.5e-5
TRAIN_WEIGHT_DECAY = 0.01
TRAIN_WARMUP_UPDATES = 20000
TRAIN_GRAD_ACCUMULATION_STEPS = 1
TRAIN_MAX_GRAD_NORM = 1.0
TRAIN_BATCH_SIZE_PER_GPU = 4096
TRAIN_MAX_SAMPLES = 32
TRAIN_SAVE_PER_UPDATES = 10000
TRAIN_LAST_PER_UPDATES = 1000
TRAIN_KEEP_LAST = 2

BENCHMARK_DEVICE = "cuda"
BENCHMARK_BATCH_SIZE = 1
BENCHMARK_FRAME_LENGTH = 256
BENCHMARK_LONG_FRAME_LENGTH = 1024
BENCHMARK_TEXT_LENGTH = 80
BENCHMARK_SAMPLE_STEPS = 4
BENCHMARK_WARMUP_ITERS = 1
BENCHMARK_ITERS = 3

INFER_SEED = 0
INFER_NFE_STEP = 16
EVAL_GPU_LIST = "[0,1]"


In [ ]:
# Cell 2: Helper dan path
import csv
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path


def clean_str(value: str) -> str:
    return (value or "").strip()


def run(cmd, cwd=None, env=None, capture_output=False):
    printable = cmd if isinstance(cmd, str) else " ".join(str(x) for x in cmd)
    print(f"$ {printable}")
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
        text=True,
        capture_output=capture_output,
    )


def build_runtime_env() -> dict:
    env = os.environ.copy()
    site_packages = sorted((VENV_DIR / "lib").glob("python*/site-packages"))
    if site_packages:
        sp = site_packages[-1]
        cuda_rel = [
            "nvidia/cublas/lib",
            "nvidia/cuda_runtime/lib",
            "nvidia/cudnn/lib",
            "nvidia/cufft/lib",
            "nvidia/nccl/lib",
            "nvidia/nvjitlink/lib",
        ]
        cuda_libs = [str(sp / rel) for rel in cuda_rel if (sp / rel).exists()]
        if cuda_libs:
            current = env.get("LD_LIBRARY_PATH", "")
            env["LD_LIBRARY_PATH"] = ":".join(cuda_libs + ([current] if current else []))
    env["PYTHONNOUSERSITE"] = "1"
    env["CUDA_VISIBLE_DEVICES"] = "0,1"
    env["TOKENIZERS_PARALLELISM"] = "false"
    env["PYTHONFAULTHANDLER"] = "1"
    env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,roundup_power2_divisions:16"
    env["TORCH_ALLOW_TF32_CUBLAS_OVERRIDE"] = "1"
    env["WANDB_MODE"] = "disabled"
    env.pop("PYTHONPATH", None)
    return env


def resolve_checkpoint(save_dir: Path) -> Path:
    model_last = save_dir / "model_last.pt"
    if model_last.exists():
        return model_last
    candidates = sorted(save_dir.glob("model_*.pt"))
    if candidates:
        return candidates[-1]
    raise FileNotFoundError(f"Checkpoint tidak ditemukan di {save_dir}")


KAGGLE_WORKING = Path("/kaggle/working")
KAGGLE_TEMP = Path("/kaggle/temp")
REPO_DIR = KAGGLE_WORKING / REPO_NAME
VENV_DIR = KAGGLE_TEMP / "f5tts-benchmark-venv"
PYTHON311_BIN = Path("/usr/bin/python3.11")
PYTHON_BIN = VENV_DIR / "bin/python"

DATASET_ROOT = Path(DATASET_ROOT_RAW).expanduser()
CSV_1 = Path(CSV_1_RAW).expanduser() if clean_str(CSV_1_RAW) else None
CSV_2 = Path(CSV_2_RAW).expanduser() if clean_str(CSV_2_RAW) else None
WAVLM_CKPT = Path(WAVLM_CKPT_RAW).expanduser() if clean_str(WAVLM_CKPT_RAW) else None
HF_TOKEN = clean_str(HF_TOKEN_RAW)

CONFIG_DIR = REPO_DIR / "src/f5_tts/configs"
BASELINE_CONFIG_NAME = "traineo1_kaggle_benchmark_baseline.yaml"
HYBRID_CONFIG_NAME = "traineo1_kaggle_benchmark_hybrid.yaml"
BASELINE_CONFIG_PATH = CONFIG_DIR / BASELINE_CONFIG_NAME
HYBRID_CONFIG_PATH = CONFIG_DIR / HYBRID_CONFIG_NAME

BASELINE_SAVE_DIR_REL = f"ckpts/{NOTEBOOK_TAG}_baseline_{DATASET_NAME}"
HYBRID_SAVE_DIR_REL = f"ckpts/{NOTEBOOK_TAG}_hybrid_{DATASET_NAME}"
BASELINE_SAVE_DIR = REPO_DIR / BASELINE_SAVE_DIR_REL
HYBRID_SAVE_DIR = REPO_DIR / HYBRID_SAVE_DIR_REL

PREPARED_DATASET_DIR = REPO_DIR / "data" / f"{DATASET_NAME}_pinyin"
MERGED_CSV = KAGGLE_WORKING / f"{DATASET_NAME}_merged.csv"
RESULTS_ROOT = KAGGLE_WORKING / f"{NOTEBOOK_TAG}_outputs"
GENERATED_ROOT = RESULTS_ROOT / "generated"
SUBJECTIVE_ROOT = RESULTS_ROOT / "subjective"
MATRIX_ROOT = RESULTS_ROOT / "matrix"
BENCHMARK_JSON = RESULTS_ROOT / "benchmark_hybrid_vs_baseline.json"

print("REPO_DIR         :", REPO_DIR)
print("VENV_DIR         :", VENV_DIR)
print("BASELINE_SAVE_DIR:", BASELINE_SAVE_DIR)
print("HYBRID_SAVE_DIR  :", HYBRID_SAVE_DIR)
print("RESULTS_ROOT     :", RESULTS_ROOT)


In [ ]:
# Cell 3: Clone repo
KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)
KAGGLE_TEMP.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

if REPO_DIR.exists():
    print("Repo sudah ada, skip clone:", REPO_DIR)
else:
    run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])

run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"])


In [ ]:
# Cell 4: Install Python 3.11 + venv + torch
run(["apt-get", "update"])
run(["apt-get", "install", "-y", "python3.11", "python3.11-venv", "python3.11-dev", "build-essential", "git"])

if not VENV_DIR.exists():
    run([str(PYTHON311_BIN), "-m", "venv", str(VENV_DIR)])
else:
    print("Venv sudah ada:", VENV_DIR)

run([str(PYTHON_BIN), "-m", "pip", "install", "--upgrade", "pip", "wheel", "setuptools<82"])
run([
    str(PYTHON_BIN),
    "-m",
    "pip",
    "install",
    "--index-url",
    "https://download.pytorch.org/whl/cu128",
    "torch==2.8.0+cu128",
    "torchvision==0.23.0+cu128",
    "torchaudio==2.8.0+cu128",
])


In [ ]:
# Cell 5: Install repo, eval deps, mamba, flash-attn (hard fail kalau gagal)
runtime_env = build_runtime_env()

run([str(PYTHON_BIN), "-m", "pip", "install", "-e", ".[eval]"], cwd=REPO_DIR, env=runtime_env)
run([str(PYTHON_BIN), "-m", "pip", "install", "ninja", "huggingface_hub", "ctranslate2==4.5.0"], cwd=REPO_DIR, env=runtime_env)

CAUSAL_CONV1D_WHL = "https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/causal_conv1d-1.6.1+cu12torch2.8cxx11abiTRUE-cp311-cp311-linux_x86_64.whl"
MAMBA_SSM_WHL = "https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+cu12torch2.8cxx11abiTRUE-cp311-cp311-linux_x86_64.whl"
FLASH_ATTN_WHL = "https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/flash_attn-2.8.3+cu12torch2.8cxx11abiTRUE-cp311-cp311-linux_x86_64.whl"

run([str(PYTHON_BIN), "-m", "pip", "uninstall", "-y", "mamba-ssm", "causal-conv1d", "flash-attn", "flash_attn"], cwd=REPO_DIR, env=runtime_env)
run([str(PYTHON_BIN), "-m", "pip", "install", "--no-deps", "--force-reinstall", CAUSAL_CONV1D_WHL], cwd=REPO_DIR, env=runtime_env)
run([str(PYTHON_BIN), "-m", "pip", "install", "--no-deps", "--force-reinstall", MAMBA_SSM_WHL], cwd=REPO_DIR, env=runtime_env)
run([str(PYTHON_BIN), "-m", "pip", "install", "--no-deps", "--force-reinstall", FLASH_ATTN_WHL], cwd=REPO_DIR, env=runtime_env)

run([
    str(PYTHON_BIN),
    "-c",
    "import torch, accelerate, mamba_ssm, flash_attn; "
    "print('torch', torch.__version__); "
    "print('accelerate', accelerate.__version__); "
    "print('mamba', mamba_ssm.__version__); "
    "print('flash_attn', flash_attn.__version__)"
], cwd=REPO_DIR, env=runtime_env)


In [ ]:
# Cell 6: Validasi 2x GPU tanpa fallback
runtime_env = build_runtime_env()
result = run([
    str(PYTHON_BIN),
    "-c",
    "import torch; "
    "print('gpu_count =', torch.cuda.device_count()); "
    "[print('gpu', i, torch.cuda.get_device_name(i)) for i in range(torch.cuda.device_count())]"
], cwd=REPO_DIR, env=runtime_env, capture_output=True)
print(result.stdout)

gpu_lines = [line.strip() for line in result.stdout.splitlines() if line.strip()]
if not any(line.startswith('gpu_count =') for line in gpu_lines):
    raise RuntimeError("Gagal membaca jumlah GPU")
gpu_count = int([line for line in gpu_lines if line.startswith('gpu_count =')][0].split('=')[1].strip())
if gpu_count < 2:
    raise RuntimeError(f"Notebook ini wajib 2 GPU. Terdeteksi cuma {gpu_count}.")

run([
    str(PYTHON_BIN),
    "-c",
    "import flash_attn, mamba_ssm, torch; "
    "assert torch.cuda.device_count() >= 2; "
    "print('dual_gpu_ok'); print('flash_attn_ok'); print('mamba_ok')"
], cwd=REPO_DIR, env=runtime_env)


In [ ]:
# Cell 7: Prepare metadata, vocab, dataset, dan asset evaluasi
from itertools import chain
from huggingface_hub import hf_hub_download

if not DATASET_ROOT.exists():
    raise FileNotFoundError(f"Dataset tidak ditemukan: {DATASET_ROOT}")

if CSV_1 is None:
    found = sorted(DATASET_ROOT.glob("**/metadata.csv"))
    CSV_1 = found[0] if found else None
if CSV_1 is None:
    raise FileNotFoundError("metadata.csv tidak ditemukan. Isi CSV_1_RAW.")
if CSV_2 is not None and not CSV_2.exists():
    raise FileNotFoundError(f"CSV_2 tidak ditemukan: {CSV_2}")

def load_rows(csv_path: Path):
    rows = []
    with open(csv_path, "r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.reader(handle, delimiter="|")
        first_row = next(reader, None)
        if first_row is None:
            return rows
        has_header = len(first_row) >= 2 and first_row[0].strip() == "audio_file" and first_row[1].strip() == "text"
        iterator = reader if has_header else chain([first_row], reader)
        for row in iterator:
            if len(row) < 2:
                continue
            audio_file = row[0].strip()
            text = row[1].strip()
            if audio_file and text:
                rows.append({"audio_file": audio_file, "text": text})
    return rows

def resolve_audio_path(raw_path: str, csv_path: Path) -> Path:
    candidate = Path(raw_path).expanduser()
    if candidate.is_absolute() and candidate.exists():
        return candidate.resolve()
    for base in [csv_path.parent, DATASET_ROOT]:
        full_path = (Path(base) / raw_path).expanduser()
        if full_path.exists():
            return full_path.resolve()
    return (DATASET_ROOT / raw_path).expanduser().resolve()

merged_rows = []
seen_audio = set()
for csv_path in [CSV_1, CSV_2]:
    if csv_path is None:
        continue
    for row in load_rows(Path(csv_path)):
        audio_path = resolve_audio_path(row["audio_file"], Path(csv_path))
        audio_key = str(audio_path)
        if audio_key in seen_audio:
            continue
        seen_audio.add(audio_key)
        merged_rows.append({"audio_file": audio_key, "text": row["text"]})

if not merged_rows:
    raise RuntimeError("Metadata kosong setelah digabung.")

with open(MERGED_CSV, "w", encoding="utf-8", newline="") as handle:
    writer = csv.writer(handle, delimiter="|", quoting=csv.QUOTE_MINIMAL)
    writer.writerow(["audio_file", "text"])
    for row in merged_rows:
        writer.writerow([row["audio_file"], row["text"]])

print("Merged rows:", len(merged_rows))
print("Merged CSV :", MERGED_CSV)

vocab_path = REPO_DIR / "data" / "Emilia_ZH_EN_pinyin" / "vocab.txt"
vocab_path.parent.mkdir(parents=True, exist_ok=True)
if not vocab_path.exists() or vocab_path.stat().st_size == 0:
    downloaded = hf_hub_download(repo_id="SWivid/F5-TTS", filename="F5TTS_v1_Base/vocab.txt", token=HF_TOKEN or None)
    shutil.copy2(downloaded, vocab_path)
print("Vocab ready:", vocab_path)

prepared_ok = (
    (PREPARED_DATASET_DIR / "raw.arrow").exists()
    and (PREPARED_DATASET_DIR / "duration.json").exists()
    and (PREPARED_DATASET_DIR / "vocab.txt").exists()
)
if not prepared_ok:
    run([
        str(PYTHON_BIN),
        "src/f5_tts/train/datasets/prepare_csv_wavs.py",
        str(MERGED_CSV),
        str(PREPARED_DATASET_DIR),
        "--workers",
        str(TRAIN_NUM_WORKERS),
    ], cwd=REPO_DIR, env=build_runtime_env())
print("Prepared dataset:", PREPARED_DATASET_DIR)

if WAVLM_CKPT is None or not WAVLM_CKPT.exists():
    raise FileNotFoundError("Isi WAVLM_CKPT_RAW dengan checkpoint WavLM lokal untuk SIM-o.")
print("WavLM checkpoint:", WAVLM_CKPT)

if not EVAL_SPECS:
    raise ValueError("EVAL_SPECS kosong. Isi minimal satu task evaluasi.")
for spec in EVAL_SPECS:
    meta_file = clean_str(spec.get("meta_file", ""))
    if not meta_file:
        raise ValueError(f"meta_file kosong untuk task {spec['name']}")
    if not Path(meta_file).exists():
        raise FileNotFoundError(f"meta_file tidak ditemukan: {meta_file}")
    if spec["task_type"] == "librispeech":
        test_clean = clean_str(spec.get("librispeech_test_clean_path") or "")
        if not test_clean or not Path(test_clean).exists():
            raise FileNotFoundError(f"librispeech_test_clean_path tidak ditemukan untuk {spec['name']}")
print("Eval assets siap.")


In [ ]:
# Cell 8: Tulis runtime configs baseline vs hybrid
from omegaconf import OmegaConf

CONFIG_DIR.mkdir(parents=True, exist_ok=True)

baseline_cfg = OmegaConf.load(REPO_DIR / "src/f5_tts/configs/F5TTS_v1_Base.yaml")
baseline_cfg.model.name = "F5TTS_v1_Base_Kaggle_Benchmark_Baseline"
baseline_cfg.datasets.name = DATASET_NAME
baseline_cfg.datasets.batch_size_per_gpu = TRAIN_BATCH_SIZE_PER_GPU
baseline_cfg.datasets.batch_size_type = "frame"
baseline_cfg.datasets.max_samples = TRAIN_MAX_SAMPLES
baseline_cfg.datasets.num_workers = TRAIN_NUM_WORKERS
baseline_cfg.optim.epochs = TRAIN_EPOCHS
baseline_cfg.optim.learning_rate = TRAIN_LR
baseline_cfg.optim.weight_decay = TRAIN_WEIGHT_DECAY
baseline_cfg.optim.num_warmup_updates = TRAIN_WARMUP_UPDATES
baseline_cfg.optim.grad_accumulation_steps = TRAIN_GRAD_ACCUMULATION_STEPS
baseline_cfg.optim.max_grad_norm = TRAIN_MAX_GRAD_NORM
baseline_cfg.optim.mixed_precision = ACCELERATE_MIXED_PRECISION
baseline_cfg.model.arch.attn_backend = "torch"
baseline_cfg.model.arch.checkpoint_activations = True
baseline_cfg.ckpts.logger = None
baseline_cfg.ckpts.log_samples = False
baseline_cfg.ckpts.save_per_updates = TRAIN_SAVE_PER_UPDATES
baseline_cfg.ckpts.last_per_updates = TRAIN_LAST_PER_UPDATES
baseline_cfg.ckpts.keep_last_n_checkpoints = TRAIN_KEEP_LAST
baseline_cfg.ckpts.save_dir = BASELINE_SAVE_DIR_REL
baseline_cfg.ckpts.student_init_checkpoint = PRETRAIN_CKPT
baseline_cfg.ckpts.load_ema_student_init = True
OmegaConf.save(baseline_cfg, BASELINE_CONFIG_PATH)

hybrid_cfg = OmegaConf.load(REPO_DIR / "src/f5_tts/configs/F5TTS_v1_Base_Mamba_Conservative.yaml")
hybrid_cfg.model.name = "F5TTS_v1_Base_Kaggle_Benchmark_Hybrid"
hybrid_cfg.datasets.name = DATASET_NAME
hybrid_cfg.datasets.batch_size_per_gpu = TRAIN_BATCH_SIZE_PER_GPU
hybrid_cfg.datasets.batch_size_type = "frame"
hybrid_cfg.datasets.max_samples = TRAIN_MAX_SAMPLES
hybrid_cfg.datasets.num_workers = TRAIN_NUM_WORKERS
hybrid_cfg.optim.epochs = TRAIN_EPOCHS
hybrid_cfg.optim.learning_rate = TRAIN_LR
hybrid_cfg.optim.weight_decay = TRAIN_WEIGHT_DECAY
hybrid_cfg.optim.num_warmup_updates = TRAIN_WARMUP_UPDATES
hybrid_cfg.optim.grad_accumulation_steps = TRAIN_GRAD_ACCUMULATION_STEPS
hybrid_cfg.optim.max_grad_norm = TRAIN_MAX_GRAD_NORM
hybrid_cfg.optim.mixed_precision = ACCELERATE_MIXED_PRECISION
hybrid_cfg.model.arch.attn_backend = "flash_attn"
hybrid_cfg.model.arch.checkpoint_activations = True
hybrid_cfg.ckpts.logger = None
hybrid_cfg.ckpts.log_samples = False
hybrid_cfg.ckpts.save_per_updates = TRAIN_SAVE_PER_UPDATES
hybrid_cfg.ckpts.last_per_updates = TRAIN_LAST_PER_UPDATES
hybrid_cfg.ckpts.keep_last_n_checkpoints = TRAIN_KEEP_LAST
hybrid_cfg.ckpts.save_dir = HYBRID_SAVE_DIR_REL
hybrid_cfg.ckpts.teacher_checkpoint = PRETRAIN_CKPT
hybrid_cfg.ckpts.student_init_checkpoint = PRETRAIN_CKPT
hybrid_cfg.ckpts.load_ema_teacher = True
hybrid_cfg.ckpts.load_ema_student_init = True
hybrid_cfg.model.distill.enabled = True
hybrid_cfg.model.distill.hidden_weight = 0.02
hybrid_cfg.model.distill.output_weight = 0.05
hybrid_cfg.model.distill.hidden_layers = [11, 14, 21]
OmegaConf.save(hybrid_cfg, HYBRID_CONFIG_PATH)

print("Baseline config:", BASELINE_CONFIG_PATH)
print("Hybrid config  :", HYBRID_CONFIG_PATH)


In [ ]:
# Cell 9: Train baseline di 2x T4
train_env = build_runtime_env()
train_env["OMP_NUM_THREADS"] = str(TRAIN_NUM_WORKERS)
train_env["MKL_NUM_THREADS"] = str(TRAIN_NUM_WORKERS)

if (BASELINE_SAVE_DIR / "model_last.pt").exists():
    print("Baseline checkpoint sudah ada, train.py akan auto-resume.")
else:
    print("Baseline training mulai dari pretrained:", PRETRAIN_CKPT)

baseline_cmd = [
    str(PYTHON_BIN),
    "-m",
    "accelerate.commands.launch",
    f"--num_processes={ACCELERATE_NUM_PROCESSES}",
    f"--mixed_precision={ACCELERATE_MIXED_PRECISION}",
    "--dynamo_backend=no",
    "src/f5_tts/train/train.py",
    "--config-name",
    BASELINE_CONFIG_NAME,
]
run(baseline_cmd, cwd=REPO_DIR, env=train_env)
run(["ls", "-lah", str(BASELINE_SAVE_DIR)], cwd=REPO_DIR, env=train_env)


In [ ]:
# Cell 10: Train hybrid (flash-attn wajib aktif) di 2x T4
train_env = build_runtime_env()
train_env["OMP_NUM_THREADS"] = str(TRAIN_NUM_WORKERS)
train_env["MKL_NUM_THREADS"] = str(TRAIN_NUM_WORKERS)

run([
    str(PYTHON_BIN),
    "-c",
    "import flash_attn; print('flash_attn runtime ok:', flash_attn.__version__)"
], cwd=REPO_DIR, env=train_env)

if (HYBRID_SAVE_DIR / "model_last.pt").exists():
    print("Hybrid checkpoint sudah ada, train.py akan auto-resume.")
else:
    print("Hybrid training mulai dari pretrained teacher/student:", PRETRAIN_CKPT)

hybrid_cmd = [
    str(PYTHON_BIN),
    "-m",
    "accelerate.commands.launch",
    f"--num_processes={ACCELERATE_NUM_PROCESSES}",
    f"--mixed_precision={ACCELERATE_MIXED_PRECISION}",
    "--dynamo_backend=no",
    "src/f5_tts/train/train.py",
    "--config-name",
    HYBRID_CONFIG_NAME,
]
run(hybrid_cmd, cwd=REPO_DIR, env=train_env)
run(["ls", "-lah", str(HYBRID_SAVE_DIR)], cwd=REPO_DIR, env=train_env)


In [ ]:
# Cell 11: Benchmark compute baseline torch vs hybrid flash-attn
benchmark_cmd = [
    str(PYTHON_BIN),
    "src/f5_tts/scripts/benchmark_hybrid_mamba.py",
    "--device", BENCHMARK_DEVICE,
    "--batch-size", str(BENCHMARK_BATCH_SIZE),
    "--frame-length", str(BENCHMARK_FRAME_LENGTH),
    "--long-frame-length", str(BENCHMARK_LONG_FRAME_LENGTH),
    "--text-length", str(BENCHMARK_TEXT_LENGTH),
    "--sample-steps", str(BENCHMARK_SAMPLE_STEPS),
    "--warmup-iters", str(BENCHMARK_WARMUP_ITERS),
    "--iters", str(BENCHMARK_ITERS),
    "--baseline-attn-backend", "torch",
    "--hybrid-attn-backend", "flash_attn",
]
bench = run(benchmark_cmd, cwd=REPO_DIR, env=build_runtime_env(), capture_output=True)
print(bench.stdout)
BENCHMARK_JSON.write_text(bench.stdout, encoding="utf-8")
print("Saved benchmark JSON:", BENCHMARK_JSON)


In [ ]:
# Cell 12: Batch inference untuk baseline dan hybrid
baseline_ckpt = resolve_checkpoint(BASELINE_SAVE_DIR)
hybrid_ckpt = resolve_checkpoint(HYBRID_SAVE_DIR)
print("Baseline ckpt:", baseline_ckpt)
print("Hybrid ckpt  :", hybrid_ckpt)

def infer_one_model(model_label: str, config_path: Path, ckpt_path: Path, spec: dict):
    model_out_dir = GENERATED_ROOT / model_label / spec["name"]
    model_out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        str(PYTHON_BIN),
        "-m",
        "accelerate.commands.launch",
        f"--num_processes={ACCELERATE_NUM_PROCESSES}",
        f"--mixed_precision={ACCELERATE_MIXED_PRECISION}",
        "--dynamo_backend=no",
        "src/f5_tts/eval/eval_infer_batch.py",
        "--config_path", str(config_path),
        "--ckpt_path", str(ckpt_path),
        "--output_dir", str(model_out_dir),
        "--meta_file", spec["meta_file"],
        "-s", str(INFER_SEED),
        "-n", config_path.stem,
        "-c", "0",
        "-nfe", str(INFER_NFE_STEP),
        "-t", spec["name"],
    ]
    if spec["task_type"] == "librispeech":
        cmd.extend(["-p", str(spec["librispeech_test_clean_path"])])
    run(cmd, cwd=REPO_DIR, env=build_runtime_env())
    return model_out_dir

generated_dirs = {"baseline": {}, "hybrid": {}}
for spec in EVAL_SPECS:
    generated_dirs["baseline"][spec["name"]] = infer_one_model("baseline", BASELINE_CONFIG_PATH, baseline_ckpt, spec)
    generated_dirs["hybrid"][spec["name"]] = infer_one_model("hybrid", HYBRID_CONFIG_PATH, hybrid_ckpt, spec)

print(json.dumps({k: {kk: str(vv) for kk, vv in v.items()} for k, v in generated_dirs.items()}, indent=2))


In [ ]:
# Cell 13: Objective eval + subjective pack + matrix per task
def run_objective_eval(spec: dict, gen_dir: Path):
    if spec["task_type"] == "seedtts":
        base_cmd = [
            str(PYTHON_BIN),
            "src/f5_tts/eval/eval_seedtts_testset.py",
            "--meta_file", spec["meta_file"],
            "--wavlm_ckpt_dir", str(WAVLM_CKPT),
            "-l", spec["lang"],
            "-g", str(gen_dir),
            "-n", EVAL_GPU_LIST,
        ]
        run(base_cmd + ["-e", "wer"], cwd=REPO_DIR, env=build_runtime_env())
        run(base_cmd + ["-e", "sim"], cwd=REPO_DIR, env=build_runtime_env())
    else:
        base_cmd = [
            str(PYTHON_BIN),
            "src/f5_tts/eval/eval_librispeech_test_clean.py",
            "--meta_file", spec["meta_file"],
            "--wavlm_ckpt_dir", str(WAVLM_CKPT),
            "-g", str(gen_dir),
            "-n", EVAL_GPU_LIST,
            "-p", str(spec["librispeech_test_clean_path"]),
        ]
        run(base_cmd + ["-e", "wer"], cwd=REPO_DIR, env=build_runtime_env())
        run(base_cmd + ["-e", "sim"], cwd=REPO_DIR, env=build_runtime_env())

for spec in EVAL_SPECS:
    base_dir = GENERATED_ROOT / "baseline" / spec["name"]
    hybrid_dir = GENERATED_ROOT / "hybrid" / spec["name"]

    run_objective_eval(spec, base_dir)
    run_objective_eval(spec, hybrid_dir)

    subjective_dir = SUBJECTIVE_ROOT / spec["name"]
    prep_cmd = [
        str(PYTHON_BIN),
        "src/f5_tts/scripts/prepare_subjective_eval.py",
        "--baseline-dir", str(base_dir),
        "--hybrid-dir", str(hybrid_dir),
        "--task", spec["task_type"],
        "--meta-file", spec["meta_file"],
        "--output-dir", str(subjective_dir),
        "--baseline-name", "baseline_f5",
        "--hybrid-name", "hybrid_f5_mamba",
    ]
    if spec["task_type"] == "librispeech":
        prep_cmd.extend(["--librispeech-test-clean-path", str(spec["librispeech_test_clean_path"])])
    run(prep_cmd, cwd=REPO_DIR, env=build_runtime_env())

    matrix_dir = MATRIX_ROOT / spec["name"]
    build_cmd = [
        str(PYTHON_BIN),
        "src/f5_tts/scripts/build_eval_matrix.py",
        "--baseline-dir", str(base_dir),
        "--hybrid-dir", str(hybrid_dir),
        "--output-dir", str(matrix_dir),
        "--baseline-name", "baseline_f5",
        "--hybrid-name", "hybrid_f5_mamba",
    ]
    run(build_cmd, cwd=REPO_DIR, env=build_runtime_env())


In [ ]:
                # Cell 14: Tampilkan ringkasan matrix evaluasi
                all_rows = []
                for spec in EVAL_SPECS:
                    matrix_json = MATRIX_ROOT / spec["name"] / "evaluation_matrix.json"
                    if matrix_json.exists():
                        rows = json.loads(matrix_json.read_text(encoding="utf-8"))
                        for row in rows:
                            row = dict(row)
                            row["task"] = spec["name"]
                            all_rows.append(row)

                print(json.dumps(all_rows, indent=2, ensure_ascii=False))
                print("
Matrix root   :", MATRIX_ROOT)
                print("Subjective root:", SUBJECTIVE_ROOT)
                print("Benchmark json:", BENCHMARK_JSON)
                print("
Untuk SMOS/CMOS:")
                print("- isi kolom smos_score pada smos_manifest.csv")
                print("- isi kolom cmos_score pada cmos_blind_manifest.csv")
                print("- lalu rerun build_eval_matrix.py dengan --smos-ratings / --cmos-ratings / --cmos-key")


## Notes

- Notebook ini sengaja **tidak fallback** ke single GPU atau `torch` backend untuk hybrid.
- Baseline tetap `torch` attention, hybrid tetap `flash_attn`.
- Training dua model dijalankan **sequential**, tapi masing-masing memakai `accelerate --num_processes=2` untuk 2x T4.
- `WER` dan `SIM-o` berjalan otomatis.
- `SMOS` dan `CMOS` disiapkan sebagai manifest evaluasi manusia karena memang tidak valid kalau dipalsukan secara otomatis.
